# RQ2 - Du bao AR vs Random Walk

AR(p) tren tung nhan to, do tre chon tren tap train bang BIC (AIC la do nhay), danh gia expanding-window ngoai mau; kiem dinh Diebold-Mariano co hieu chinh Harvey-Leybourne-Newbold.

In [1]:
import sys
sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src import config, data_fetch, utils, interactive as ia
pd.options.display.float_format = '{:.4f}'.format
from src import forecasting as fc

## 1. Nap nhan to tu RQ1

In [2]:
factors_monthly = utils.ensure_monthly_freq(pd.read_csv('../data/ns_factors_monthly.csv', index_col=0))
print(config.TRAIN_TEST_SPLIT_DATE, config.AR_LAG_IC)
factors_monthly.tail()

2015-12-31 bic


,level,slope,curvature,rmse,n_maturities
date,,,,,
2026-04,5.0446,-1.1893,-2.3826,0.1214,10
2026-05,5.0296,-1.1996,-1.8929,0.1185,10
2026-06,4.9318,-0.8967,-1.5813,0.1190,10
2026-07,5.3054,-1.3540,-1.5406,0.1217,10
2026-08,5.2633,-1.2766,-1.3058,0.1150,10


## 2. AR vs RW cho tung nhan to (train <= 2015-12, test tu 2016-01)

In [3]:
results = [fc.run_ar_vs_rw_pipeline(factors_monthly, col, rolling=True) for col in ['level','slope','curvature']]
summary = pd.concat(results, ignore_index=True)
summary

,factor,horizon,ar_lag,RMSE_AR,RMSE_RW,MAE_AR,MAE_RW,n_obs,DM_stat,DM_pvalue
0,level,1,1,0.2304,0.2286,0.1816,0.1804,128,0.7504,0.4544
1,level,3,1,0.4141,0.4077,0.3199,0.3061,126,0.6056,0.5459
2,level,6,1,0.5670,0.5502,0.4595,0.4311,123,0.5950,0.5530
3,level,12,1,0.7857,0.7642,0.6442,0.6193,117,0.2999,0.7648
4,slope,1,1,0.3131,0.3106,0.2323,0.2318,128,0.7639,0.4463
5,slope,3,1,0.5809,0.5713,0.4401,0.4264,126,0.6873,0.4931
6,slope,6,1,0.8864,0.8697,0.6624,0.6644,123,0.4408,0.6601
7,slope,12,1,1.3990,1.3918,1.0493,1.0888,117,0.0826,0.9343
8,curvature,1,1,0.7005,0.7050,0.5099,0.5137,128,-0.3538,0.7241
9,curvature,3,1,1.1811,1.2036,0.9110,0.9271,126,-0.5075,0.6127


## 3. Doc ket qua
- `RMSE_AR < RMSE_RW`: AR tot hon ve diem so. `DM_pvalue < 0.05`: chenh lech co y nghia thong ke (DM co hieu chinh HLN, t(n-1)).
- Ky vong: AR khong thang RW co y nghia thong ke o bat ky nhan to/horizon nao tren mau 2016-2026; curvature la nhan to gan nhat.

In [4]:
summary.assign(improve_pct=(1-summary.RMSE_AR/summary.RMSE_RW)*100)[['factor','horizon','ar_lag','RMSE_AR','RMSE_RW','improve_pct','DM_pvalue']]

,factor,horizon,ar_lag,RMSE_AR,RMSE_RW,improve_pct,DM_pvalue
0,level,1,1,0.2304,0.2286,-0.7715,0.4544
1,level,3,1,0.4141,0.4077,-1.5615,0.5459
2,level,6,1,0.5670,0.5502,-3.0667,0.5530
3,level,12,1,0.7857,0.7642,-2.8130,0.7648
4,slope,1,1,0.3131,0.3106,-0.8138,0.4463
5,slope,3,1,0.5809,0.5713,-1.6877,0.4931
6,slope,6,1,0.8864,0.8697,-1.9198,0.6601
7,slope,12,1,1.3990,1.3918,-0.5193,0.9343
8,curvature,1,1,0.7005,0.7050,0.6292,0.7241
9,curvature,3,1,1.1811,1.2036,1.8655,0.6127


## 4. Du bao tai tung ky han (dung lai duong cong tu 3 nhan to)

In [5]:
codes = list(config.NS_SERIES)
yields = data_fetch.fetch_fred_data(use_cache=True)[0]
ym = yields[codes].resample('ME').last()
by_mat = fc.run_yield_curve_forecast(factors_monthly, ym, config.NS_SERIES)
by_mat.groupby('horizon').agg(RMSE_AR=('RMSE_AR','mean'), RMSE_RW=('RMSE_RW','mean'), n_AR_better=('improve_pct', lambda x: int((x>0).sum())), n_sig=('DM_pvalue', lambda x: int((x<0.05).sum())))

,RMSE_AR,RMSE_RW,n_AR_better,n_sig
horizon,,,,
1,0.2528,0.2299,1,5
3,0.4881,0.4640,4,2
6,0.7543,0.7230,3,1
12,1.2059,1.2040,4,0


## 5. Xuat bang

In [6]:
from pathlib import Path
out_path = Path('../data/ar_vs_rw_results.csv')
summary.to_csv(out_path, index=False)
by_mat.to_csv('../outputs/rq2_yield_forecast_by_maturity.csv', index=False)
summary.to_csv('../outputs/rq2_ar_vs_rw.csv', index=False)
print('Da ghi', len(summary), 'dong')

Da ghi 12 dong


## 6. Tuong tac: RMSE theo ky han va horizon

In [7]:
import plotly.express as px
d = by_mat.assign(horizon=by_mat.horizon.astype(str) + 'm')
fig = px.line(d, x='tau_years', y='improve_pct', color='horizon', markers=True, hover_data=['maturity','DM_pvalue'],
              title='Cai thien RMSE cua AR so voi RW (%) theo ky han (duong > 0: AR tot hon)')
fig.add_hline(y=0, line_dash='dash'); fig.update_layout(template='plotly_white', xaxis_title='Ky han (nam)'); fig.show()